# Final Notebook - Classifying Spam Emails

Notebook cu?i n?y t?ng h?p n?i dung t? c?c notebook `00` ??n `06` v? ph?n giao di?n ki?m tra spam ?? th?m v?o project.

M?c ti?u c?a notebook:

- T?m t?t b?i to?n ph?n lo?i email spam/not spam.
- Tr?nh b?y ngu?n d? li?u, ch?t l??ng d? li?u v? c?n b?ng nh?n.
- M? t? preprocessing, feature engineering v? c?c model ?? train.
- Hi?n th? k?t qu? ??nh gi?, learning curve v? confusion matrix.
- Demo d? ?o?n email m?i b?ng model ?? train.
- Ghi l?i c?ch ch?y giao di?n web local.

## 1. Problem Definition

B?i to?n: x?y d?ng h? th?ng ph?n lo?i email th?nh 2 l?p:

- `0`: not spam / ham
- `1`: spam

Input l? n?i dung email d?ng text. Output l? nh?n d? ?o?n `spam` ho?c `not spam`, k?m `spam_score` v? c?c t?/c?m t? ?nh h??ng t?i k?t qu?.

V?i b?i to?n spam detection, nh?m kh?ng ch? nh?n accuracy. C?c ch? s? quan tr?ng g?m:

- `precision_spam`: khi model b?o spam th? ??ng bao nhi?u.
- `recall_spam`: model b?t ???c bao nhi?u email spam th?t.
- `f1_spam`: c?n b?ng gi?a precision v? recall.
- `false_positive_rate`: t? l? ch?n nh?m email t?t.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'

print('Project root:', PROJECT_ROOT)
print('Reports exists:', REPORTS_DIR.exists())
print('Models exists:', MODELS_DIR.exists())

## 2. Data Collection

Ngu?n d? li?u ch?nh c?a project:

| Ngu?n | Vai tr? |
| --- | --- |
| SpamAssassin Public Corpus | Ngu?n ham/spam public, c? nh?n r? |
| SetFit Enron Spam | Dataset email spam/ham t? Hugging Face |
| locuoco 300k spam/ham/phish | Dataset l?n, map spam/phish v? class spam |
| TREC Spam Track | Ngu?n ch?nh th?ng ?? tham kh?o th?m |
| CMU Enron Email Dataset | Ngu?n email Enron ch?nh th?ng, c?n g?n nh?n c?n th?n n?u m? r?ng |

Project chu?n h?a d? li?u v? schema:

```text
source,file_name,label,label_name,subject,text
```

In [ ]:
source_path = PROJECT_ROOT / 'data_sources' / 'data_links.csv'
if source_path.exists():
    sources = pd.read_csv(source_path)
    display(sources)
else:
    print('Kh?ng t?m th?y data_sources/data_links.csv')

## 3. Data Quality v? c?n b?ng d? li?u

Sau khi g?p d? li?u, project ki?m tra:

- Label kh?ng h?p l?.
- Text r?ng.
- Text qu? ng?n.
- D?ng tr?ng l?p theo `label` + `text`.

Sau khi l?c l?i, dataset ???c c?n b?ng l?i ?? s? l??ng spam v? not spam ngang nhau. Vi?c c?n b?ng gi?p model kh?ng thi?n l?ch v? class c? s? l??ng l?n h?n.

In [ ]:
quality_path = REPORTS_DIR / 'data_quality_report.json'
loader_path = REPORTS_DIR / 'data_loader_report.json'

for path in [loader_path, quality_path]:
    print('\n---', path.name, '---')
    if path.exists():
        payload = json.loads(path.read_text(encoding='utf-8'))
        print(json.dumps(payload, indent=2, ensure_ascii=False))
    else:
        print('Kh?ng t?m th?y file report:', path)

## 4. Preprocessing

Module ch?nh: `src/text_preprocess.py`

C?c b??c x? l? text:

- X?a HTML, script, style.
- Chu?n h?a URL th?nh `urltoken`.
- Chu?n h?a email address th?nh `emailtoken`.
- Chu?n h?a s? th?nh `numbertoken`.
- Lowercase.
- Lo?i k? t? ??c bi?t.
- Lo?i stopwords ti?ng Anh n?u c? NLTK corpus, fallback n?u thi?u.

In [ ]:
from src.text_preprocess import clean_email_text

examples = [
    '<html>FREE prize!!! Click https://spam.example now, contact test@example.com for 100 dollars</html>',
    'Hi team, please confirm tomorrow meeting agenda and send the project report when ready.',
]

for text in examples:
    print('RAW:', text)
    print('CLEAN:', clean_email_text(text))
    print()

## 5. Feature Engineering v? Model Training

Model ???c train b?ng pipeline:

```text
TfidfVectorizer(max_features=25000, ngram_range=(1, 2), stop_words='english')
-> classifier
```

Ba model so s?nh:

- Multinomial Naive Bayes
- Logistic Regression
- Linear SVM

Model t?t nh?t ???c l?u t?i `models/spam_classifier.joblib`.

In [ ]:
metrics_path = REPORTS_DIR / 'model_metrics.csv'
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics)

    metric_cols = ['accuracy', 'precision_spam', 'recall_spam', 'f1_spam']
    ax = metrics.set_index('model')[metric_cols].plot(kind='bar', figsize=(10, 5), ylim=(0.9, 1.0))
    ax.set_title('Model metrics comparison')
    ax.set_ylabel('Score')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print('Kh?ng t?m th?y reports/model_metrics.csv')

## 6. Confusion Matrix

C?c ?nh d??i ??y l? k?t qu? ??nh gi? model ?? train, ???c sinh t? `src/model_evaluate.py` v? l?u trong `reports/figures/`.

![Linear SVM confusion matrix](../reports/figures/linear_svm_confusion_matrix.png)

![Logistic Regression confusion matrix](../reports/figures/logistic_regression_confusion_matrix.png)

![Naive Bayes confusion matrix](../reports/figures/naive_bayes_confusion_matrix.png)

## 7. Learning Curve

C?c ?nh learning curve d??i ??y l? k?t qu? ?? train v? ?? ???c l?y l?i t? qu? tr?nh ??nh gi? model. ??y kh?ng ph?i ?nh m?u.

- ???ng ??: F1-score tr?n t?p train.
- ???ng xanh: F1-score cross-validation.
- Kho?ng c?ch gi?a hai ???ng c?ng nh? th? model c?ng ?t overfitting.

![Learning Curve - Linear SVM](../reports/figures/learning_curve_linear_svm.png)

![Learning Curve - Logistic Regression](../reports/figures/learning_curve_logistic_regression.png)

![Learning Curve - Naive Bayes](../reports/figures/learning_curve_naive_bayes.png)

## 8. Demo Predict Email m?i

Module ch?nh: `src/predict.py`

H?m `predict_email()` tr? v?:

- `prediction`: `spam` ho?c `not spam`
- `spam_score`: ?i?m nghi?ng v? spam trong kho?ng 0-1
- `cleaned_text`: text sau ti?n x? l?

In [ ]:
from src.predict import load_pipeline, predict_email

model_path = MODELS_DIR / 'spam_classifier.joblib'
model = load_pipeline(model_path)

demo_emails = [
    'Congratulations winner, claim your free lottery prize money now by clicking this urgent link.',
    'Hi team, please confirm tomorrow meeting agenda and send the project report when ready.',
]

for email_text in demo_emails:
    result = predict_email(email_text, model=model)
    print('Email:', email_text)
    print('Prediction:', result['prediction'])
    print('Spam score:', result['spam_score'])
    print('Cleaned:', result['cleaned_text'])
    print()

## 9. Web UI ki?m tra spam

Project c? giao di?n Flask t?i `app.py`.

C?ch ch?y:

```powershell
python app.py
```

Sau ?? m?:

```text
http://127.0.0.1:5000
```

L?u ?: kh?ng m? tr?c ti?p file `templates/spam_checker.html` b?ng Live Server port `5500`, v? Flask c?n render c? ph?p Jinja nh? `{% ... %}` v? `{{ ... }}`.

In [ ]:
sample_dir = PROJECT_ROOT / 'sample_emails'
if sample_dir.exists():
    for path in sorted(sample_dir.glob('*.txt')):
        print(path.name)
else:
    print('Kh?ng t?m th?y sample_emails/')

## 10. K?t lu?n

Pipeline cu?i c?ng ?? c? ?? c?c ph?n:

- Ngu?n d? li?u v? ki?m tra ngu?n.
- Data loading, quality check v? c?n b?ng nh?n.
- Text preprocessing.
- Feature engineering b?ng TF-IDF.
- Train v? so s?nh Naive Bayes, Logistic Regression, Linear SVM.
- ??nh gi? b?ng accuracy, precision, recall, F1-score v? confusion matrix.
- D? ?o?n email m?i b?ng model ?? train.
- Giao di?n web local ?? ki?m tra email spam/not spam.

Khi thuy?t tr?nh, nh?m n?n nh?n m?nh r?ng k?t qu? d? ?o?n d?a tr?n ??c tr?ng v?n b?n ?? h?c t? dataset, kh?ng ph?i rule th? c?ng.